# 04 — Penyesuaian profil Vs dengan HVSR

**Alur:** `profil Vs dari dispersi + HVSR teramati → uji bentuk eliptisitas → kandidat profil Vs akhir`.

**Input:** profil dan fit dari notebook 03, kurva serta QC SESAME dari notebook 01, dan parameter pencarian lokal pada sel di bawah.

**Proses:** bandingkan bentuk HVSR dengan eliptisitas Rayleigh teoretis pada frekuensi yang sama; ubah profil dalam batas kecil sambil menjaga kecocokan dispersi. Periksa puncak, lembah, keandalan data, dan dukungan kedalaman 0–30 m sebelum menyatakan hasil final.

**Output:** kandidat, grafik fit, dan alasan keputusan di `outputs/<site_id>/04/`. Profil Vs dan Vs30 final hanya diterbitkan bila seluruh gerbang ilmiah lulus. Notebook 05 membaca statusnya untuk pembandingan.

**Acuan refinement:** [Zor et al. (2010), §5.1](https://academic.oup.com/gji/article/182/3/1603/600175). Tahap MAM sebelumnya mengikuti Hayashi (2022); Hayashi juga membahas HVSR pada §9/Figure 10. Adaptasi ini menjaga fit dispersi sambil menilai bentuk HVSR, tanpa menyamakan amplitudo HVSR dengan eliptisitas.


## Input pengguna — pencarian lokal dan gerbang interpretasi

`reviewed` dan `depth_30m_supported` tetap `False` sampai ada penilaian ilmiah. Bila Anda mengubahnya menjadi `True`, isi catatan bukti yang bersesuaian. Nilai rasio Poisson, densitas, dan batas lapisan diambil dari keluaran notebook 03 agar tidak perlu dimasukkan ulang.

In [ ]:
SITE_ID = "solo_pilot"
REFINEMENT_PARAMETERS = {
    "random_seed": 2030,
    "candidate_count": 128,
    "relative_perturbation_std": 0.08,
    "minimum_shape_correlation_gain": 0.02,
    "maximum_dispersion_rmse_increase_m_s": 3.0,
    "reviewed": False,
    "depth_30m_supported": False,
}
REFINEMENT_REVIEW_NOTE = ""
DEPTH_30M_EVIDENCE_NOTE = ""

In [ ]:
from __future__ import annotations

import hashlib
import json
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

from mhvsr_vs30.mam.inversion import forward_rayleigh_phase, vs30_from_layers
from mhvsr_vs30.mam.refinement import forward_ellipticity, log_shape_correlation, refine_candidate

ROOT=Path.cwd()
PARAM=REFINEMENT_PARAMETERS
SITE=SITE_ID
if PARAM['reviewed'] and not REFINEMENT_REVIEW_NOTE.strip():
    raise ValueError('Add REFINEMENT_REVIEW_NOTE before marking refinement reviewed')
if PARAM['depth_30m_supported'] and not DEPTH_30M_EVIDENCE_NOTE.strip():
    raise ValueError('Add DEPTH_30M_EVIDENCE_NOTE before marking depth supported')
P01=ROOT/'outputs'/SITE/'01'
P03=ROOT/'outputs'/SITE/'03'
OUT=ROOT/'outputs'/SITE/'04'
OUT.mkdir(parents=True,exist_ok=True)
meta01=json.loads((P01/'preprocessing_metadata.json').read_text(encoding='utf-8'))
meta03=json.loads((P03/'inversion_status.json').read_text(encoding='utf-8'))
if meta03.get('run_state')!='complete' or meta03.get('method_revision')!='hayashi_2022_v1':
    raise ValueError('Notebook 03 must complete with the current method')
for artifact,expected in ((ROOT/meta03['source_csv'],meta03['source_sha256']),
    (ROOT/'outputs'/SITE/'02'/'dispersion_qc.json',meta03['upstream_qc_sha256'])):
    if hashlib.sha256(artifact.read_bytes()).hexdigest()!=expected:
        raise ValueError('Dispersion input changed: rerun notebooks 03-04')
source_level='final' if meta03['mode']=='final' else 'preview'
MODEL_PATH=P03/source_level/'vs_dispersion_best.csv'
FIT_PATH=P03/source_level/'dispersion_fit.csv'
for artifact,key in ((MODEL_PATH,'model_sha256'),(FIT_PATH,'fit_sha256')):
    if hashlib.sha256(artifact.read_bytes()).hexdigest()!=meta03[key]:
        raise ValueError('Inversion artifact changed: rerun notebook 03')
(OUT/'refinement_status.json').write_text(json.dumps({'run_state':'running',
    'method_revision':'hayashi_2022_zor_2010_v1','vs30_reported':False}),encoding='utf-8')
for name in ('vs_final_best.csv','vs30_results.csv'):
    old=OUT/name
    if old.exists():
        archive=OUT/'superseded'/datetime.now(UTC).strftime('%Y%m%dT%H%M%S%fZ')
        archive.mkdir(parents=True,exist_ok=True)
        old.replace(archive/name)
model_df=pd.read_csv(MODEL_PATH)
dispersion=pd.read_csv(FIT_PATH).sort_values('period_s')
hvsr=pd.read_csv(P01/'hvsr_observed.csv')
assert len(model_df)>=2 and len(dispersion)>=8
print('Source:',source_level,'| HVSR flags:',meta01['hvsr_qc_flags'])

## 1. Pita frekuensi bersama dan asumsi model

Perbandingan hanya memakai frekuensi yang tersedia pada dispersi dan HVSR. Pada pilot ini pita dispersi sekitar 11–30 Hz; fitur HVSR di luar pita tersebut tidak dapat diuji terhadap model dispersi. Rasio Poisson, densitas, dan batas model mengikuti asumsi notebook 03, bukan hasil ukur tersendiri.

In [ ]:
n_finite=len(model_df)-1
base=np.r_[model_df.thickness_m.iloc[:n_finite].to_numpy(dtype=float),
           model_df.vs_m_s.to_numpy(dtype=float)]
period_d=dispersion.period_s.to_numpy(dtype=float)
velocity_d=dispersion.observed_velocity_m_s.to_numpy(dtype=float)
f_low=float(dispersion.frequency_hz.min())
f_high=float(dispersion.frequency_hz.max())
h=hvsr.loc[hvsr.frequency_hz.between(f_low,f_high) &
           hvsr.hvsr_mean.gt(0)].sort_values('frequency_hz',ascending=False)
assert len(h)>=8
period_h=1/h.frequency_hz.to_numpy(dtype=float)
observed_h=h.hvsr_mean.to_numpy(dtype=float)
poisson=float(meta03['poisson_ratio_assumed'])
density=float(meta03['density_assumed_g_cm3'])
bounds=[tuple(map(float,b)) for b in meta03['layer_bounds']]
assert len(bounds)==len(base)
baseline_ell=forward_ellipticity(period_h,base[:n_finite],base[n_finite:],
                                  poisson=poisson,density_g_cm3=density)
baseline_corr=log_shape_correlation(observed_h,baseline_ell)
baseline_disp=forward_rayleigh_phase(period_d,base[:n_finite],base[n_finite:],
                                     poisson=poisson,density_g_cm3=density)
baseline_rmse=float(np.sqrt(np.mean((baseline_disp-velocity_d)**2)))
print('Overlap:',f_low,'-',f_high,'Hz | HVSR points:',len(h),
      '| shape correlation:',baseline_corr,'| dispersion RMSE:',baseline_rmse)

## 2. Cari perubahan kecil tanpa merusak kecocokan dispersi

Kandidat diambil dengan seed tetap di sekitar model inversi. Korelasi log bentuk HVSR dan eliptisitas dibandingkan tanpa pergeseran frekuensi. Hasil tetap preview bila metadata geometri, jam, mode, atau kejelasan puncak belum lulus review.


In [ ]:
best,rows=refine_candidate(base,bounds,period_d,velocity_d,period_h,observed_h,
    poisson=poisson,density_g_cm3=density,seed=int(PARAM['random_seed']),
    candidate_count=int(PARAM['candidate_count']),
    perturbation_std=float(PARAM['relative_perturbation_std']),
    max_rmse_increase_m_s=float(PARAM['maximum_dispersion_rmse_increase_m_s']))
scores=pd.DataFrame(rows)
scores.to_csv(OUT/'candidate_scores.csv',index=False)
finite_scores=scores.loc[np.isfinite(scores.shape_correlation) & scores.accepted_dispersion]
shape_informative=bool(len(finite_scores) and np.isfinite(baseline_corr))
gain=(float(finite_scores.shape_correlation.max()-baseline_corr) if shape_informative else float('-inf'))
improved=bool(gain>=float(PARAM['minimum_shape_correlation_gain']))
selected=best if improved else base
selected_ell=forward_ellipticity(period_h,selected[:n_finite],selected[n_finite:],
                                  poisson=poisson,density_g_cm3=density)
selected_corr=log_shape_correlation(observed_h,selected_ell)
selected_disp=forward_rayleigh_phase(period_d,selected[:n_finite],selected[n_finite:],
                                     poisson=poisson,density_g_cm3=density)
selected_rmse=float(np.sqrt(np.mean((selected_disp-velocity_d)**2)))
print('Valid candidate count:',int(scores.accepted_dispersion.sum()),
      '| correlation gain:',gain,'| selected:',improved,
      '| selected dispersion RMSE:',selected_rmse)


## 3. Simpan kandidat, fit, dan status ilmiah

`vs_refined_candidate.csv` adalah kandidat yang dapat ditinjau. Berkas `vs_final_best.csv` dan `vs30_results.csv` hanya ditulis setelah input dispersi direview, optimasi konvergen, HVSR jelas, geometri/jam/mode terverifikasi, refinement disetujui, dan cakupan kedalaman 30 m didukung. Nilai Vs30 matematis dari model saja belum membuktikan resolusi 30 m.


In [ ]:
th=selected[:n_finite]; vs=selected[n_finite:]
vp=vs*np.sqrt(2*(1-poisson)/(1-2*poisson))
profile=pd.DataFrame({'layer_index':np.arange(1,len(vs)+1),
    'top_depth_m':np.r_[0,np.cumsum(th)],
    'bottom_depth_m':np.r_[np.cumsum(th),np.nan],
    'thickness_m':np.r_[th,np.nan],
    'vs_m_s':vs,'vp_m_s':vp,'density_g_cm3':density,'poisson_assumed':poisson})
profile.to_csv(OUT/'vs_refined_candidate.csv',index=False)
pd.DataFrame({'frequency_hz':h.frequency_hz.to_numpy(),
    'hvsr_observed':observed_h,'ellipticity_baseline':baseline_ell,
    'ellipticity_refined':selected_ell}).to_csv(OUT/'hvsr_shape_fit.csv',index=False)
dfit=dispersion.copy()
dfit['velocity_refined_m_s']=selected_disp
dfit['residual_refined_m_s']=selected_disp-velocity_d
dfit.to_csv(OUT/'dispersion_fit_refined.csv',index=False)

ordered=h.assign(ellipticity_refined=selected_ell).sort_values('frequency_hz')
ordered_frequency=ordered.frequency_hz.to_numpy(dtype=float)
observed_log=np.log(ordered.hvsr_mean.to_numpy(dtype=float))
theory_log=np.log(ordered.ellipticity_refined.to_numpy(dtype=float))
observed_peaks=ordered_frequency[find_peaks(observed_log,prominence=0.08)[0]].tolist()
observed_troughs=ordered_frequency[find_peaks(-observed_log,prominence=0.08)[0]].tolist()
theory_peaks=ordered_frequency[find_peaks(theory_log,prominence=0.08)[0]].tolist()
theory_troughs=ordered_frequency[find_peaks(-theory_log,prominence=0.08)[0]].tolist()
peak_trough_consistent=(len(observed_peaks)==len(theory_peaks) and
    len(observed_troughs)==len(theory_troughs) and
    all(abs(a-b)/a<=0.1 for a,b in zip(observed_peaks,theory_peaks,strict=True)) and
    all(abs(a-b)/a<=0.1 for a,b in zip(observed_troughs,theory_troughs,strict=True)))

def standardized_log(values):
    values=np.log(np.asarray(values,dtype=float))
    spread=values.std()
    return (values-values.mean())/spread if spread>=0.01 else np.zeros_like(values)

fig,axes=plt.subplots(1,3,figsize=(14,4.5))
axes[0].semilogx(h.frequency_hz,standardized_log(observed_h),'ko-',ms=3,label='Observed HVSR')
axes[0].semilogx(h.frequency_hz,standardized_log(baseline_ell),'--',label='Before')
axes[0].semilogx(h.frequency_hz,standardized_log(selected_ell),label='After')
axes[0].set(xlabel='Frequency (Hz)',ylabel='Standardized log shape',
            title=f'Shape r {baseline_corr:.2f} → {selected_corr:.2f}')
axes[0].legend(fontsize=7)
axes[1].plot(dispersion.frequency_hz,velocity_d,'ko',ms=3,label='Input')
axes[1].plot(dispersion.frequency_hz,baseline_disp,'--',label='Before')
axes[1].plot(dispersion.frequency_hz,selected_disp,label='After')
axes[1].set(xlabel='Frequency (Hz)',ylabel='Phase speed (m/s)',
            title=f'RMSE {baseline_rmse:.1f} → {selected_rmse:.1f} m/s')
axes[1].legend(fontsize=7)
for parameters,label,style in ((base,'Before','--'),(selected,'After','-')):
    edges=np.r_[0,np.cumsum(parameters[:n_finite]),40.0]
    speeds=np.r_[parameters[n_finite:],parameters[-1]]
    axes[2].step(speeds,edges,where='pre',linestyle=style,label=label)
axes[2].invert_yaxis()
axes[2].set(xlabel='Vs (m/s)',ylabel='Depth (m)',ylim=(40,0),title='Candidate Vs')
axes[2].legend(fontsize=7)
for ax in axes: ax.grid(alpha=0.2)
fig.tight_layout(); fig.savefig(OUT/'refinement_qc.png',dpi=160); plt.show()

requirements={
    'shape_comparison_informative':shape_informative and bool(np.isfinite(selected_corr)),
    'dispersion_input_reviewed':meta03['mode']=='final',
    'inversion_converged':any(meta03.get('run_converged',[])),
    'hvsr_clarity_passed':not meta01['hvsr_qc_flags'],
    'geometry_verified':meta03['upstream_verification_flags']['geometry_verified'],
    'physical_clock_drift_verified':meta03['upstream_verification_flags']['physical_clock_drift_verified'],
    'wavefield_mode_verified':meta03['upstream_verification_flags']['wavefield_mode_verified'],
    'wavelength_range_reviewed':meta03['upstream_verification_flags'].get('wavelength_range_reviewed',False),
    'refinement_reviewed':bool(PARAM['reviewed']),
    'depth_30m_supported':bool(PARAM['depth_30m_supported']),
    'dispersion_preserved':selected_rmse<=baseline_rmse+PARAM['maximum_dispersion_rmse_increase_m_s'],
    'peak_trough_consistent':peak_trough_consistent,
}
can_report_final=all(requirements.values())
if can_report_final:
    profile.to_csv(OUT/'vs_final_best.csv',index=False)
    pd.DataFrame([{'site_id':SITE,'vs30_m_s':vs30_from_layers(th,vs),
                   'status':'reviewed_final_model'}]).to_csv(OUT/'vs30_results.csv',index=False)
status={'run_state':'complete','method_revision':'hayashi_2022_zor_2010_v1',
    'refinement_reference':'Zor et al. (2010), Section 5.1',
    'dispersion_reference_doi':'10.1007/s10950-021-10051-y',
    'inversion_status_sha256':hashlib.sha256((P03/'inversion_status.json').read_bytes()).hexdigest(),
    'site_id':SITE,'created_at_utc':datetime.now(UTC).isoformat(),
    'input_model_level':source_level,'hvsr_qc_flags':meta01['hvsr_qc_flags'],
    'frequency_overlap_hz':[f_low,f_high],'hvsr_points_used':len(h),
    'baseline_shape_correlation':baseline_corr if np.isfinite(baseline_corr) else None,
    'selected_shape_correlation':selected_corr if np.isfinite(selected_corr) else None,
    'shape_comparison_status':'informative' if shape_informative else 'flat_or_noninformative',
    'baseline_dispersion_rmse_m_s':baseline_rmse,
    'selected_dispersion_rmse_m_s':selected_rmse,
    'candidate_count':int(PARAM['candidate_count']),'selected_perturbation':improved,
    'observed_peak_hz':observed_peaks,'observed_trough_hz':observed_troughs,
    'theory_peak_hz':theory_peaks,'theory_trough_hz':theory_troughs,
    'final_requirements':requirements,'final_vs_reported':can_report_final,
    'vs30_reported':can_report_final,
    'vs30_results_sha256':(hashlib.sha256((OUT/'vs30_results.csv').read_bytes()).hexdigest() if can_report_final else None),
    'final_profile_sha256':(hashlib.sha256((OUT/'vs_final_best.csv').read_bytes()).hexdigest() if can_report_final else None),
    'comparison_basis':'shape_of_log_curves_not_absolute_amplitude',
    'model_source_sha256':hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest(),
    'hvsr_source_sha256':hashlib.sha256((P01/'hvsr_observed.csv').read_bytes()).hexdigest(),
    'config_sha256':hashlib.sha256(json.dumps(PARAM,sort_keys=True).encode()).hexdigest(),
    'refinement_review_note':REFINEMENT_REVIEW_NOTE,
    'depth_30m_evidence_note':DEPTH_30M_EVIDENCE_NOTE}
(OUT/'refinement_status.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
print('Final Vs reported:',can_report_final,'| candidate:',OUT/'vs_refined_candidate.csv')